# Bi-directional Recurrent Neural Network Example

Xây dựng mạng RNN 2 chiều với PyTorch

## Tổng quan về BiRNN

<img src="https://www.easy-tensorflow.com/images/NN/03.png" alt="nn" style="width: 600px;"/>


## Tổng quan về bộ dữ liệu MNIST

Ví dụ này sử dụng bộ dữ liệu về chữ số viết tay MNIST. Bộ dữ liệu chữa 60k mẫu cho huấn luyện và 10k mẫu cho kiểm thử.

![MNIST Dataset](https://i1.wp.com/varianceexplained.org/images/mnist.png?w=456)

Để phân loại hình ảnh sử dụng RNN, chúng ta sẽ coi mỗi hàng là 1 chuỗi pixels. Bởi vì kích thước ảnh là 28*28px, ta sẽ sử lý 28 chuỗi của 28 timesteps cho tất cả các sample.

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import numpy as np

In [30]:
# Chuẩn bị dữ liệu
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Chuyển đổi sang định dạng float32.
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)
x_train, x_test = x_train.reshape([-1, 28, 28]), x_test.reshape([-1, 28, 28])
# Chuẩn hóa ảnh từ from [0, 255] to [0, 1].
x_train, x_test = x_train / 255., x_test / 255.
x_train, x_test, y_train, y_test = torch.from_numpy(x_train), torch.from_numpy(x_test), torch.from_numpy(y_train).type(torch.LongTensor), torch.from_numpy(y_test).type(torch.LongTensor)

In [31]:
trainloader = []
batch_size = 64  # Increased batch size for mini-batch training
for (i,j) in zip(x_train, y_train):
    trainloader.append([i,j])
trainloader = torch.utils.data.DataLoader(trainloader, shuffle=True, batch_size=batch_size)

testloader = []
for (i,j) in zip(x_test, y_test):
    testloader.append([i,j])
testloader = torch.utils.data.DataLoader(testloader, shuffle=True, batch_size=batch_size)

In [32]:
    def forward(self, x):
        
        # Khởi tạo hidden and cell state (layer_dim * 2 for bidirectional, x.size(0) for batch_size)
        h0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim)
        c0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim)
        
        # One time step
        out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # Take the last timestep output, apply dropout, and pass through FC
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        
        return out

In [33]:
# Create BiLSTM
input_dim = 28    # chiều của input
hidden_dim = 128  # Increased hidden state
layer_dim = 2     # Increased layers
output_dim = 10   # chiều của vector output

model = BiRNNModel(input_dim, hidden_dim, layer_dim, output_dim)

# Cross Entropy Loss 
criterion = nn.CrossEntropyLoss()

# Use Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [34]:
for epoch in range(10):  # Increased epochs

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        # zero the parameter gradients
        optimizer.zero_grad()
        
        # inputs is already (batch, 28, 28), labels (batch,)
        
        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:    # print every 100 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 100))
            running_loss = 0.0

print('Finished Training')

[1,   100] loss: 1.650
[1,   200] loss: 0.950
[1,   300] loss: 0.728
[1,   400] loss: 0.531
[1,   500] loss: 0.458
[1,   600] loss: 0.395
[1,   700] loss: 0.353
[1,   800] loss: 0.320
[1,   900] loss: 0.318
[2,   100] loss: 0.263
[2,   200] loss: 0.246
[2,   300] loss: 0.249
[2,   400] loss: 0.205
[2,   500] loss: 0.229
[2,   600] loss: 0.204
[2,   700] loss: 0.183
[2,   800] loss: 0.189
[2,   900] loss: 0.186
[3,   100] loss: 0.166
[3,   200] loss: 0.174
[3,   300] loss: 0.155
[3,   400] loss: 0.142
[3,   500] loss: 0.153
[3,   600] loss: 0.147
[3,   700] loss: 0.145
[3,   800] loss: 0.190
[3,   900] loss: 0.149
[4,   100] loss: 0.132
[4,   200] loss: 0.144
[4,   300] loss: 0.116
[4,   400] loss: 0.122
[4,   500] loss: 0.123
[4,   600] loss: 0.136
[4,   700] loss: 0.136
[4,   800] loss: 0.133
[4,   900] loss: 0.117
[5,   100] loss: 0.128
[5,   200] loss: 0.114
[5,   300] loss: 0.112
[5,   400] loss: 0.109
[5,   500] loss: 0.111
[5,   600] loss: 0.098
[5,   700] loss: 0.111
[5,   800] 

In [35]:
correct = 0
total = 0
# quá trình kiểm thử ko cần thiết phải tính gradients cho output
with torch.no_grad():
    for data in testloader:
        images, labels = data
        
        # calculate outputs by running images through the network
        outputs = model(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

Accuracy of the network on the 10000 test images: 97 %
